Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\colab\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [5]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [6]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [7]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [8]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [9]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [10]:
datosNormalizados.shape

(43800, 6)

In [11]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [12]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [13]:
futuros = 1
pasados  = 12

In [14]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [15]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 6)
Dimensiones de Y: (43788, 1)


In [16]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894 ]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347 ]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449]
 [ 0.57004095 -0.65625711 -1.34931411  0.90832835 -0.38094383 -0.09555657]]


Se dividen nuevamente los conjuntos de datos

In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 12, 6)
Las dimensiones de testX son:  (8801, 12, 6)
Las dimensiones de valX son:  (4336, 12, 6)


In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [19]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [20]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [21]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [22]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

958/958 - 36s - 37ms/step - ia: 0.5232 - loss: 0.5913 - mae: 0.5389 - rmse: 0.7432 - smape: 1.0938 - val_ia: 0.3560 - val_loss: 0.3671 - val_mae: 0.4190 - val_rmse: 0.5169 - val_smape: 0.8776

Epoch 2/128                                           

958/958 - 16s - 16ms/step - ia: 0.6335 - loss: 0.4389 - mae: 0.4574 - rmse: 0.6422 - smape: 0.9077 - val_ia: 0.4006 - val_loss: 0.2832 - val_mae: 0.3781 - val_rmse: 0.4683 - val_smape: 0.8222

Epoch 3/128                                           

958/958 - 12s - 13ms/step - ia: 0.6695 - loss: 0.3782 - mae: 0.4224 - rmse: 0.5955 - smape: 0.8432 - val_ia: 0.4333 - val_loss: 0.2456 - val_mae: 0.3491 - val_rmse: 0.4355 - val_smape: 0.7644

Epoch 4/128                                           

958/958 - 12s - 13ms/step - ia: 0.6918 - loss: 0.3431 - mae: 0.3985 - rmse: 0.5655 - smape: 0.8035 - val_ia: 0.4746 - val_loss: 0.2108 - val_mae: 0.3175 - val_rmse: 0.4032 - val_smape: 0.7083

Epoc

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                           

120/120 - 38s - 314ms/step - ia: 0.5226 - loss: 0.6517 - mae: 0.5162 - rmse: 0.7597 - smape: 1.0757 - val_ia: 0.6801 - val_loss: 0.2621 - val_mae: 0.3510 - val_rmse: 0.4798 - val_smape: 0.7603

Epoch 2/16                                                                           

120/120 - 35s - 292ms/step - ia: 0.7860 - loss: 0.2076 - mae: 0.2989 - rmse: 0.4479 - smape: 0.6524 - val_ia: 0.7434 - val_loss: 0.1601 - val_mae: 0.2720 - val_rmse: 0.3781 - val_smape: 0.6276

Epoch 3/16                                                                           

120/120 - 23s - 188ms/step - ia: 0.8481 - loss: 0.1157 - mae: 0.2189 - rmse: 0.3356 - smape: 0.5130 - val_ia: 0.8486 - val_loss: 0.0738 - val_mae: 0.1671 - val_rmse: 0.2566 - val_smape: 0.4308

Epoch 4/16                                                                           

120/120 - 20s - 170ms/step - ia: 0.8761 - loss: 0.0831 - mae: 0.1804 -

...

Epoch 1/64                                                                            

479/479 - 12s - 26ms/step - ia: 0.7913 - loss: 0.1884 - mae: 0.2838 - rmse: 0.4144 - smape: 0.6071 - val_ia: 0.7574 - val_loss: 0.0762 - val_mae: 0.1805 - val_rmse: 0.2494 - val_smape: 0.4408

Epoch 2/64                                                                            

479/479 - 4s - 8ms/step - ia: 0.8432 - loss: 0.1162 - mae: 0.2226 - rmse: 0.3309 - smape: 0.4907 - val_ia: 0.8099 - val_loss: 0.0627 - val_mae: 0.1496 - val_rmse: 0.2198 - val_smape: 0.3833

Epoch 3/64                                                                            

479/479 - 5s - 11ms/step - ia: 0.8503 - loss: 0.1085 - mae: 0.2130 - rmse: 0.3206 - smape: 0.4668 - val_ia: 0.8158 - val_loss: 0.0602 - val_mae: 0.1441 - val_rmse: 0.2173 - val_smape: 0.3756

Epoch 4/64                                                                            

479/479 - 5s - 11ms/step - ia: 0.8537 - loss: 0.1043 - mae: 0.2080 - rms

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/32                                                                            

3832/3832 - 87s - 23ms/step - ia: 0.6931 - loss: 0.3197 - mae: 0.3630 - rmse: 0.4903 - smape: 0.7445 - val_ia: 0.3635 - val_loss: 0.1214 - val_mae: 0.2265 - val_rmse: 0.2708 - val_smape: 0.5390

Epoch 2/32                                                                            

3832/3832 - 71s - 19ms/step - ia: 0.7753 - loss: 0.1736 - mae: 0.2758 - rmse: 0.3722 - smape: 0.5966 - val_ia: 0.4101 - val_loss: 0.0966 - val_mae: 0.1979 - val_rmse: 0.2421 - val_smape: 0.4793

Epoch 3/32                                                                            

3832/3832 - 73s - 19ms/step - ia: 0.8018 - loss: 0.1357 - mae: 0.2434 - rmse: 0.3283 - smape: 0.5410 - val_ia: 0.4700 - val_loss: 0.0731 - val_mae: 0.1674 - val_rmse: 0.2095 - val_smape: 0.4292

Epoch 4/32                                                                            

3832/3832 - 75s - 20ms/step - ia: 0.8160 - loss: 0.1205 - mae: 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/8                                                                             

1916/1916 - 27s - 14ms/step - ia: 0.3192 - loss: 0.8271 - mae: 0.6569 - rmse: 0.8660 - smape: 1.3995 - val_ia: 0.2431 - val_loss: 0.6415 - val_mae: 0.5707 - val_rmse: 0.6331 - val_smape: 1.1722

Epoch 2/8                                                                             

1916/1916 - 18s - 9ms/step - ia: 0.5152 - loss: 0.5941 - mae: 0.5312 - rmse: 0.7283 - smape: 1.0453 - val_ia: 0.2646 - val_loss: 0.4552 - val_mae: 0.4584 - val_rmse: 0.5232 - val_smape: 0.8696

Epoch 3/8                                                                             

1916/1916 - 17s - 9ms/step - ia: 0.6012 - loss: 0.4979 - mae: 0.4791 - rmse: 0.6648 - smape: 0.9131 - val_ia: 0.2832 - val_loss: 0.3885 - val_mae: 0.4191 - val_rmse: 0.4839 - val_smape: 0.8010

Epoch 4/8                                                                             

1916/1916 - 16s - 8ms/step - ia: 0.6285 - loss: 0.4515 - mae: 0.4

In [23]:
print(best)

{'activation': 2, 'batch': 1, 'dropout': 0.30000000000000004, 'epochs': 2, 'layers': 1.0, 'learning_rate': 0.00045371747819692945, 'units': 4}
